# 04 — Cell composition with age

Which cell types change in number with age? Fractions are computed per mouse, so each mouse counts once however many cells it gave. Then a detour: is mouse 30-M-2 sick?

Reads `data/lung_processed.h5ad`.

### 1. Setup

In [ ]:
import sys
sys.path.append("..")          # so "src" can be imported from the notebook folder

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import false_discovery_control, spearmanr
from src import aging as ag    # shared settings and functions

### 2. Load and define groups

In [ ]:
adata = sc.read_h5ad("../data/lung_processed.h5ad")
age_order = ["1m", "3m", "18m", "21m", "30m"]
meta = adata.obs[["mouse.id", "age", "sex"]].drop_duplicates().set_index("mouse.id")

### 3. How much does 30-M-2 contribute to each cell type?

If one mouse supplies most of a cell type, any "age effect" there may be that mouse.

In [ ]:
share = pd.crosstab(adata.obs["cell_ontology_class"], adata.obs["mouse.id"], normalize="index")
print(share["30-M-2"].sort_values(ascending=False).head(10).round(2))

### 4. Fraction of each cell type, per mouse

`crosstab` keeps a 0 for a mouse with none of a cell type. The `groupby(...).size()` version drops those rows and inflates the averages.

In [ ]:
cts = ["B cell", "CD8-positive, alpha-beta T cell", "CD4-positive, alpha-beta T cell",
       "NK cell", "bronchial smooth muscle cell", "classical monocyte",
       "alveolar macrophage", "fibroblast of lung"]

In [ ]:
frac = pd.crosstab(adata.obs["mouse.id"], adata.obs["cell_ontology_class"], normalize="index")
long = (frac[cts].join(meta).reset_index()
        .melt(id_vars=["mouse.id", "age", "sex"], var_name="cell_type", value_name="fraction"))
long["age"] = long["age"].astype(str)

### 5. Plot: one dot per mouse, color = sex

In [ ]:
g = sns.catplot(data=long, x="age", y="fraction", hue="sex", col="cell_type",
                col_wrap=4, kind="strip", order=age_order, sharey=False,
                height=2.8, size=7, jitter=0.15)
g.set_titles("{col_name}", size=9)
g.savefig("../figure/lung_composition_by_mouse.png", dpi=200, bbox_inches="tight")

### 6. Name the outliers

In [ ]:
print(long[long.cell_type == "classical monocyte"].sort_values("fraction", ascending=False).head())

### 7. Is 30-M-2 sick? Monocyte-lineage share per mouse

A 30-month mouse (about 80+ human years) with a huge monocyte expansion could have an infection, a tumor, or another inflammatory condition. There are no health records, so the cells' genes have to show which explanation fits.

In [ ]:
# Monocyte lineage only: narrower than ag.MYELOID, which also has alveolar macrophages
myeloid = ["classical monocyte", "intermediate monocyte", "non-classical monocyte", "lung macrophage"]
tab = pd.crosstab(adata.obs["mouse.id"], adata.obs["cell_ontology_class"])
summary = pd.DataFrame({
    "total_cells": tab.sum(axis=1),
    "myeloid_cells": tab[myeloid].sum(axis=1),
})
summary["myeloid_fraction"] = (summary.myeloid_cells / summary.total_cells).round(2)
summary = summary.join(meta).sort_values("myeloid_fraction", ascending=False)
summary.to_csv("../figure/lung_myeloid_by_mouse.csv")
summary

### 8. What state are the classical monocytes in, per mouse?

| Group | Genes | High in the outlier mouse would suggest |
| --- | --- | --- |
| Monocyte identity | *Ly6c2*, *Ccr2* | Ordinary classical monocytes |
| Interferon response | *Ifit1*, *Ifit3*, *Isg15*, *Irf7* | Antiviral-type response (e.g., infection) |
| Inflammation | *S100a8*, *S100a9*, *Il1b* | Acute inflammation or emergency myelopoiesis |
| Proliferation | *Mki67*, *Top2a* | Cells actively dividing (expansion or a myeloid tumor) |
| Blood contamination | *Hba-a1*, *Beta-s* | Lung not cleared of blood, circulating cells captured |

In [ ]:
mono = adata[adata.obs.cell_ontology_class == "classical monocyte"]
state_genes = {
    "identity": ["Ly6c2", "Ccr2"],
    "interferon": ["Ifit1", "Ifit3", "Isg15", "Irf7"],
    "inflammation": ["S100a8", "S100a9", "Il1b"],
    "proliferation": ["Mki67", "Top2a"],
    "blood": ["Hba-a1", "Beta-s"],
}
state_genes = {k: [g for g in v if g in adata.var_names] for k, v in state_genes.items()}

In [ ]:
sc.pl.dotplot(mono, state_genes, groupby="mouse.id", standard_scale="var", show=False)
plt.savefig("../figure/lung_monocyte_state.png", dpi=200, bbox_inches="tight")

### 9. What makes 30-M-2's monocytes different from other old mice's?

Exploratory only: per-cell tests overstate significance, but the top gene list is still informative.

In [ ]:
mono_old = mono[mono.obs.age == "30m"].copy()
mono_old.obs["is_30M2"] = np.where(mono_old.obs["mouse.id"] == "30-M-2", "30-M-2", "other 30m")
sc.tl.rank_genes_groups(mono_old, groupby="is_30M2", groups=["30-M-2"], reference="other 30m", method="wilcoxon")
top30 = sc.get.rank_genes_groups_df(mono_old, group="30-M-2").head(30)
top30.to_csv("../figure/lung_30M2_top_genes.csv", index=False)
top30

### 10. Is 30-M-2 different in every cell type, or only myeloid cells?

30-M-2 has few non-myeloid cells, so only read these for cell types where it has at least ~20 cells. If its B and T cells also show the signal, the condition is body-wide. If they look normal, it is specific to myeloid cells.

In [ ]:
genes_flat = [g for v in state_genes.values() for g in v]
for ct in ["B cell", "CD8-positive, alpha-beta T cell", "NK cell"]:
    sub = adata[(adata.obs.cell_ontology_class == ct) & (adata.obs.age.isin(["18m", "21m", "30m"]))]
    sc.pl.dotplot(sub, genes_flat, groupby="mouse.id", standard_scale="var", title=ct, show=False)
    name = ct.split(",")[0].replace(" ", "_")
    plt.savefig(f"../figure/lung_state_{name}.png", dpi=200, bbox_inches="tight")

### 11. Age trends without the odd samples

Rank correlation between age in months and each cell type's per-mouse fraction, leaving out samples with clearly odd composition. With about 13 mice, treat the p-values as rough. `p_bh` is corrected for testing 8 cell types.

In [ ]:
odd = ["30-M-2", "3-M-5/6", "3-M-7/8"]
rows = []
for ct in cts:
    d = long[(long.cell_type == ct) & ~long["mouse.id"].isin(odd)]
    rho, pval = spearmanr(d["age"].map(ag.AGE_MONTHS), d["fraction"])
    rows.append({"cell_type": ct, "spearman_rho": rho, "p": pval, "n_mice": len(d)})
trends = pd.DataFrame(rows).sort_values("spearman_rho")
trends["p_bh"] = false_discovery_control(trends["p"])    # Benjamini-Hochberg correction
trends = trends.round(3)
trends.to_csv("../figure/lung_composition_trends.csv", index=False)
trends

### Findings

- **30-M-2 is 97% myeloid** (7,789 of 8,014 cells). It holds 92% of all lung macrophages, 77% of intermediate monocytes, and 68% of classical monocytes. The next highest mouse is 30-M-3 at 51%.
- **Its classical monocytes are in an unusual state:** much lower *Ccr2* and lower *Ly6c2*, very high *S100a8/9*, some *Mki67* and *Top2a*, and no interferon signal.
- **Interpretation:** an expanding, inflammatory, less mature myeloid population. This fits an inflammatory condition or a myeloid disorder; the data can't say which. **Decision: exclude 30-M-2 from the myeloid cell types and from composition trends** (`ag.EXCLUDE`, `ag.main_kw`).
- **Ambient RNA:** *S100a8/9* appear in B cells of 18-M-52, 18-M-53, and 30-M-2. B cells don't make these, so the reads likely come from free-floating RNA. Handled in Phase 5.
- **The pooled 3m samples (3-M-5/6, 3-M-7/8) are outliers the other way:** ~50% fibroblasts, 2% myeloid, almost no B or CD8 T cells. Left out of the trends.
- **Trends (13 mice):** CD8 T cells rise (rho +0.70) and alveolar macrophages fall (rho −0.65). Both are about 0.06 after correction, so **suggestive, consistent with known lung immune aging**, not proven. The classical monocyte "increase" was entirely 30-M-2.